# Hartree-Fock法で原子核を解いてみよう
ここではHartree-Fock法を用いて原子核の構造を計算します．

In [ ]:
from pathlib import Path
import os, sys

REPO_NAME = "Lecture_Hokkaido"
GITHUB_REPO = "https://github.com/Takayuki-Miyagi/Lecture_Hokkaido.git"
PACKAGE_DIR = "."

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not Path(REPO_NAME).exists():
        !git clone {GITHUB_REPO}
    %cd {REPO_NAME}
    %pip install -q -e .

def find_repo_root(package_dir=PACKAGE_DIR):
    cwd = Path.cwd().resolve()

    for p in [cwd, *cwd.parents]:
        if (p / "src" / package_dir).is_dir():
            return p

    raise RuntimeError(
        f"Could not find src/{package_dir}. "
        "Please run this notebook from inside the repository."
    )

repo_root = find_repo_root()
src_dir = repo_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print("repo root:", repo_root)
print("src added:", src_dir)


import constants
import LECs
from single_partile_orbit import Orbits
from two_body_space import TwoBodySpace
from model_space import ModelSpace
from partial_wave_basis import PartialWaveChannel, PartialWaveChannels
from ho_partial_wave_basis import HOPartialWaveChannel, HOPartialWaveChannels
from nn_pot import NNPotential
from ho_nn_pot import HONNPotential
from nuclear_operator import Operator
from hartree_fock import HartreeFock


まずは計算を行うための空間を定義しましょう．

In [ ]:
emax = 6
Nmax = 2*emax
Jrel_max = 4
hw = 20.0

In [ ]:
pw = PartialWaveChannels(Jmax=Jrel_max, NMesh=40, kmax=6)

parameters = LECs.N2LO_opt
filename = "N2LOopt.npz"
vpot_mom = NNPotential(pw)
if(os.path.exists(filename)):
    vpot_mom.load_npz_into(filename)
else:
    vpot_mom.set_potential(parameters)
    vpot_mom.save_npz(filename)

In [ ]:
vpot_ho = HONNPotential(Nmax, Jrel_max, hw)
vpot_ho.set_nn_potential(vpot_mom)

In [ ]:
ms = ModelSpace()
ms.set_model_space_from_emax(emax)
ms.set_frequency(hw)
ms.set_reference_from_string("O16")
ms.assign_particle_hole()
tkin, vnn, ham = Operator(ms), Operator(ms), Operator(ms)
tkin.set_kinetic_term()
vnn.set_nn_interaction(vpot_ho)
ham = tkin + vnn
hf = HartreeFock(ham)
hf.solve()
ham = hf.transform_operator_ho_to_hf(ham)
ham.take_normal_ordering()
e2 = hf.second_order_correction(ham)
print(f"HF: {ham.get_0bme():12.6f}, 2nd order correction: {e2:12.6f}, Total: {ham.get_0bme() + e2:12.6f}")
